In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  extract_financials_repair.ipynb                                         ║
# ║  BVMT — Targeted Repair for Failed Extractions                          ║
# ║                                                                          ║
# ║  PURPOSE: Fix the ~60 records that returned conf=0.00 or                ║
# ║  total_assets:missing in the main batch. Each company has a             ║
# ║  specific problem — this notebook handles each one differently.         ║
# ║                                                                          ║
# ║  COMPANIES HANDLED:                                                      ║
# ║  A. Wrong unit (DT not detected): SOTUMAG, ADWYA, SOPAT                 ║
# ║  B. Non-standard page layout: ONE TECH, TELNET, STB, UIB                ║
# ║  C. No balance sheet in filing: HANNIBAL LEASE FY2022-2024 → skip       ║
# ║  D. Isolated failures: ICF, CELLCOM, ASSU MAGHREBIA VIE                 ║
# ║  E. SPDIT-SICAF: investment fund, different balance sheet format         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
 
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports, config, connectivity                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
import subprocess
subprocess.run(
    ['pip', 'install', 'pdfplumber', 'openai', 'psycopg2-binary',
     'python-dotenv', 'pandas', '--quiet'],
    check=False
)
 
import pdfplumber
from openai import OpenAI
import psycopg2
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os, re, json, time
 
load_dotenv()
 
PDF_BASE_DIR = Path(r'C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials')
MIN_CONFIDENCE_TO_SKIP = 0.5
DELAY = 3
 
_requests_today = 0
 
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'), port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'), user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
GPT_CLIENT = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv('GITHUB_TOKEN'),
)
GPT_MODEL = "gpt-4o"
 
# Connectivity test
try:
    _r = GPT_CLIENT.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": "Reply: READY"}],
        max_tokens=5
    )
    _requests_today += 1
    print(f"GPT-4o: {_r.choices[0].message.content.strip()}")
except Exception as e:
    print(f"ERROR: {e}")
 
print("✓ Cell 1 OK")

GPT-4o: READY! How can I
✓ Cell 1 OK


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Diagnostic: scan a PDF and show ALL pages with numbers        ║
# ║                                                                          ║
# ║  USE THIS CELL to investigate any failing PDF before fixing it.         ║
# ║  It tells you: which pages have the balance sheet, what the unit is,    ║
# ║  and shows the first 400 chars of each financial page.                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def diagnose_pdf(ticker: str, filename: str):
    """
    Scan a PDF and print a page-by-page report showing:
    - Which pages have financial keywords
    - Which pages have large numbers (6+ digits)
    - Unit declaration lines
    - First 400 chars of each page with financial content
 
    Use this to understand WHY a PDF is failing before fixing it.
    """
    safe = re.sub(r'[<>:"/\\|?*]', '_', ticker)
 
    # Try both FY_ANNUAL subfolder and root
    for subfolder in ['FY_ANNUAL', '']:
        if subfolder:
            pdf_path = PDF_BASE_DIR / safe / subfolder / filename
        else:
            pdf_path = PDF_BASE_DIR / safe / filename
        if pdf_path.exists():
            break
    else:
        print(f"FILE NOT FOUND: {ticker}/{filename}")
        return
 
    print(f"\n{'='*65}")
    print(f"DIAGNOSIS: {ticker} / {filename}")
    print(f"{'='*65}")
 
    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages : {len(pdf.pages)}")
 
        # Show unit declarations from first 6 pages
        print("\nUnit declarations (first 6 pages):")
        for i in range(min(6, len(pdf.pages))):
            text = pdf.pages[i].extract_text() or ''
            for line in text.split('\n'):
                if any(kw in line.lower() for kw in
                       ['dinar', '1.000', 'millier', 'arrondi', 'exprim', 'unit']):
                    print(f"  Page {i+1}: {line.strip()[:80]}")
 
        # Show all pages with financial content
        print("\nPages with financial content:")
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ''
            text_up = text.upper()
 
            flags = []
            if 'BILAN' in text_up:                                    flags.append('BILAN')
            if 'ACTIF' in text_up:                                    flags.append('ACTIF')
            if 'PASSIF' in text_up:                                   flags.append('PASSIF')
            if 'TOTAL' in text_up and re.search(r'\d{6,}', text):    flags.append('TOTAL+NUMS')
            if re.search(r'\bAC\s*[1-7]\b', text):                   flags.append('BCT_CODES')
            if 'PRODUIT NET' in text_up:                              flags.append('PNB')
            if re.search(r"CHIFFRE\s+D", text_up):                   flags.append('CA')
            if 'RESULTAT' in text_up and re.search(r'\d{5,}', text): flags.append('RESULTAT')
            if re.search(r'[\u0600-\u06FF]', text):                  flags.append('⚠ARABIC')
 
            if flags:
                print(f"  Page {i+1:2d}: {', '.join(flags)}")
                # Show preview of pages likely to have balance sheet
                if ('ACTIF' in flags or 'BILAN' in flags) and 'TOTAL+NUMS' in flags:
                    preview = text[:400].replace('\n', ' | ')
                    print(f"          → {preview[:200]}")
 
 
# ── Example usage — uncomment to diagnose a specific PDF ──────────────────
# diagnose_pdf('STB', 'stb_efd311216.pdf')
# diagnose_pdf('UIB', 'uib_efd311216.pdf')
# diagnose_pdf('ONE TECH HOLDING', 'oth_efd311218.pdf')
# diagnose_pdf('TELNET HOLDING', 'telnet_efd311216.pdf')
# diagnose_pdf('SOTUMAG', 'sotumag_efd311217.pdf')
 
print("✓ Cell 2 OK — diagnose_pdf() defined")
print("  Usage: diagnose_pdf('TICKER', 'filename.pdf')")
print("  Uncomment one of the examples above to inspect a failing PDF")

✓ Cell 2 OK — diagnose_pdf() defined
  Usage: diagnose_pdf('TICKER', 'filename.pdf')
  Uncomment one of the examples above to inspect a failing PDF


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Extraction helpers (prompt + API call + validation + DB)      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# These are the same helpers from the main notebook, copied here so this
# repair notebook is completely self-contained.
 
SYSTEM_PROMPT = (
    "You are a financial data extractor for Tunisian company reports. "
    "Return ONLY a valid JSON object. No markdown, no explanation, no code fences."
)
 
PROMPT_KDT = """Tunisian financial report — values in kDT. Return as-is.
 
RULES: Return ONLY valid JSON. null for missing. First (current year) column only.
total_assets = GRAND TOTAL only (not subtotals like \"Total actifs immobilisés\").
 
Balance sheet: total_assets, total_liabilities, total_loans (AC3/banks), total_deposits (PA3/banks), equity
P&L: pnb (banks), revenue (non-banks), net_result, operating_expenses
Prudential: npl_ratio (%), coverage_ratio (%), lcr (%)
 
PDF TEXT:
{text}
 
JSON: {{\"total_assets\":null,\"total_liabilities\":null,\"total_loans\":null,\"total_deposits\":null,\"equity\":null,\"pnb\":null,\"revenue\":null,\"net_result\":null,\"operating_expenses\":null,\"npl_ratio\":null,\"coverage_ratio\":null,\"lcr\":null}}"""
 
PROMPT_DT = """Tunisian financial report — values in FULL DINARS. Divide ALL monetary values by 1000.
272682126 → 272682.126 | Do NOT divide percentages (npl_ratio, coverage_ratio, lcr).
 
RULES: Return ONLY valid JSON. null for missing. First (current year) column only.
total_assets = GRAND TOTAL (skip \"Total actifs immobilisés\", \"Total actifs courants\").
 
Balance sheet (÷1000): total_assets, total_liabilities, equity, total_loans (banks), total_deposits (banks)
P&L (÷1000): pnb (banks), revenue (non-banks), net_result, operating_expenses
Prudential (no ÷): npl_ratio (%), coverage_ratio (%), lcr (%)
 
PDF TEXT:
{text}
 
JSON: {{\"total_assets\":null,\"total_liabilities\":null,\"total_loans\":null,\"total_deposits\":null,\"equity\":null,\"pnb\":null,\"revenue\":null,\"net_result\":null,\"operating_expenses\":null,\"npl_ratio\":null,\"coverage_ratio\":null,\"lcr\":null}}"""
 
 
def call_gpt(text: str, unit: str = 'kDT', retries: int = 3) -> tuple:
    """Call GPT-4o with retry. Returns (data_dict, note_string)."""
    global _requests_today
    if not text.strip():
        return {}, 'no_text'
 
    prompt = (PROMPT_KDT if unit == 'kDT' else PROMPT_DT).format(text=text)
 
    for attempt in range(1, retries + 1):
        try:
            resp = GPT_CLIENT.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=200, temperature=0.0,
            )
            _requests_today += 1
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)
            return json.loads(raw.strip()), 'OK'
 
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                if attempt < retries:
                    wait = 60 * attempt
                    print(f"\n  [Rate limit — waiting {wait}s attempt {attempt}/{retries}]",
                          end='', flush=True)
                    time.sleep(wait)
                else:
                    return {}, 'api_limit_reached'
            elif '500' in err:
                if attempt < retries:
                    time.sleep(15)
                else:
                    return {}, 'server_error'
            elif 'json' in err.lower():
                return {}, f'json_parse_error'
            else:
                if attempt < retries:
                    time.sleep(8)
                else:
                    return {}, f'error:{err[:60]}'
    return {}, 'all_retries_failed'
 
 
def validate(data: dict, company_type: str) -> tuple:
    """Validate and clean GPT-4o output. Returns (data, confidence, needs_review, notes)."""
    notes = []
    needs_review = False
    valid = 0
    MAIN = ['total_assets','total_liabilities','total_loans','total_deposits',
            'equity','pnb','revenue','net_result']
    max_kdt = 80_000_000 if company_type == 'bank' else 15_000_000
 
    for field in MAIN + ['operating_expenses']:
        val = data.get(field)
        if val is None:
            continue
        if not isinstance(val, (int, float)):
            data[field] = None; notes.append(f'{field}:not_numeric'); continue
        if abs(val) > 0 and abs(val) < 1:
            data[field] = None; notes.append(f'{field}:too_small'); continue
        if abs(val) > max_kdt:
            c = val / 1000
            if 1 <= abs(c) <= max_kdt:
                data[field] = round(c, 3)
                notes.append(f'{field}:auto_div1000({val:.0f}→{c:.0f})')
                val = c
            else:
                data[field] = None; notes.append(f'{field}:out_of_range'); continue
        if field in MAIN and data.get(field) is not None:
            valid += 1
 
    for f in ['npl_ratio','coverage_ratio','lcr']:
        val = data.get(f)
        if val is not None and (not isinstance(val,(int,float)) or not 0 < val < 300):
            data[f] = None; notes.append(f'{f}:invalid')
 
    ta, tl, eq = data.get('total_assets'), data.get('total_liabilities'), data.get('equity')
    if ta and tl and eq and ta > 0:
        diff = abs(ta - (tl + eq)) / ta
        tol = 0.05 if company_type == 'bank' else 0.25
        if diff > tol:
            needs_review = True; notes.append(f'balance_diff:{diff*100:.1f}%')
 
    if not data.get('total_assets'):
        needs_review = True; notes.append('total_assets:missing')
 
    conf = round(min(valid / len(MAIN), 1.0), 3)
    return data, conf, needs_review, ' | '.join(notes) if notes else 'OK'
 
 
def get_isin_map():
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SELECT ticker, isin_code FROM company_metadata')
            return {r[0]: r[1] for r in cur.fetchall()}
    finally:
        conn.close()
 
 
def compute_derived(data: dict, period: str) -> dict:
    net, eq = data.get('net_result'), data.get('equity')
    if net and eq and eq > 0:
        data['roe'] = round((net / eq) * (2 if 'H1' in period else 1) * 100, 4)
    pnl = data.get('pnb') or data.get('revenue')
    opex = data.get('operating_expenses')
    if opex and pnl and pnl > 0:
        data['cost_income_ratio'] = round(abs(opex) / pnl * 100, 4)
    loans, deps = data.get('total_loans'), data.get('total_deposits')
    if loans and deps and deps > 0:
        data['loan_to_deposit'] = round(loans / deps * 100, 4)
    return data
 
 
def upsert(ticker, isin, period, ped, source_pdf, company_type,
           data, confidence, needs_review, notes):
    data = compute_derived(data.copy(), period)
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                INSERT INTO financial_statements (
                    ticker, isin_code, period, period_end_date, source_pdf, company_type,
                    total_assets, total_liabilities, total_loans, total_deposits, equity,
                    pnb, revenue, net_result, operating_expenses,
                    npl_ratio, coverage_ratio, lcr,
                    roe, cost_income_ratio, loan_to_deposit,
                    extraction_confidence, needs_review, extraction_notes
                ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (ticker, period) DO UPDATE SET
                    total_assets=EXCLUDED.total_assets,
                    total_liabilities=EXCLUDED.total_liabilities,
                    total_loans=EXCLUDED.total_loans,
                    total_deposits=EXCLUDED.total_deposits,
                    equity=EXCLUDED.equity,
                    pnb=EXCLUDED.pnb, revenue=EXCLUDED.revenue,
                    net_result=EXCLUDED.net_result,
                    operating_expenses=EXCLUDED.operating_expenses,
                    npl_ratio=EXCLUDED.npl_ratio,
                    coverage_ratio=EXCLUDED.coverage_ratio,
                    lcr=EXCLUDED.lcr, roe=EXCLUDED.roe,
                    cost_income_ratio=EXCLUDED.cost_income_ratio,
                    loan_to_deposit=EXCLUDED.loan_to_deposit,
                    extraction_confidence=EXCLUDED.extraction_confidence,
                    needs_review=EXCLUDED.needs_review,
                    extraction_notes=EXCLUDED.extraction_notes,
                    scraped_at=NOW()
            ''', (ticker, isin, period, ped, source_pdf, company_type,
                  data.get('total_assets'), data.get('total_liabilities'),
                  data.get('total_loans'), data.get('total_deposits'), data.get('equity'),
                  data.get('pnb'), data.get('revenue'), data.get('net_result'),
                  data.get('operating_expenses'), data.get('npl_ratio'),
                  data.get('coverage_ratio'), data.get('lcr'),
                  data.get('roe'), data.get('cost_income_ratio'),
                  data.get('loan_to_deposit'), confidence, needs_review, notes))
        conn.commit()
        return True
    except Exception as e:
        conn.rollback()
        print(f"\n  DB error {ticker} {period}: {e}")
        return False
    finally:
        conn.close()
 
 
print("✓ Cell 3 OK — all helpers defined")

✓ Cell 3 OK — all helpers defined


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Core repair function                                          ║
# ║                                                                          ║
# ║  repair_pdf() is the main function used by all repair cells below.      ║
# ║  It differs from the main batch in 3 ways:                              ║
# ║  1. You explicitly pass unit='DT' or unit='kDT' — no auto-detection     ║
# ║     mistakes possible                                                    ║
# ║  2. You explicitly pass page_indices — send exactly the pages you know  ║
# ║     contain the balance sheet (from diagnose_pdf())                     ║
# ║  3. force=True by default — always re-extracts, ignores existing record ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def repair_pdf(ticker: str,
               filename: str,
               period: str,
               period_end_date: str,
               unit: str,
               company_type: str,
               page_indices: list,
               isin_map: dict,
               max_chars: int = 8000) -> dict:
    """
    Targeted extraction for a single PDF with explicit parameters.
 
    Args:
        ticker:          company ticker e.g. 'STB'
        filename:        PDF filename e.g. 'stb_efd311216.pdf'
        period:          e.g. 'FY 2016'
        period_end_date: e.g. '2016-12-31'
        unit:            'kDT' or 'DT' — YOU decide, no auto-detection
        company_type:    'bank', 'insurance', or 'non_bank'
        page_indices:    list of 0-based page numbers to send to GPT-4o
                         e.g. [0, 1, 2] for pages 1, 2, 3
        isin_map:        from get_isin_map()
        max_chars:       character cap on text sent to GPT-4o
    """
    safe = re.sub(r'[<>:"/\\|?*]', '_', ticker)
 
    # Find the PDF — try FY_ANNUAL subfolder first, then root
    pdf_path = None
    for subfolder in ['FY_ANNUAL', '']:
        p = PDF_BASE_DIR / safe / subfolder / filename if subfolder else PDF_BASE_DIR / safe / filename
        if p.exists():
            pdf_path = p
            break
 
    if not pdf_path:
        print(f"  ✗ FILE NOT FOUND: {ticker}/{filename}")
        return {'status': 'file_not_found'}
 
    # Extract text from the specified pages
    all_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for idx in sorted(set(page_indices)):
                if 0 <= idx < total_pages:
                    raw = pdf.pages[idx].extract_text() or ''
                    cleaned = re.sub(r'[ \t]+', ' ', raw)
                    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()
                    if cleaned:
                        all_text.append(f"[PAGE {idx+1}]\n{cleaned}")
    except Exception as e:
        print(f"  ✗ PDF read error: {e}")
        return {'status': 'read_error', 'error': str(e)}
 
    text = '\n\n'.join(all_text)
    if len(text) > max_chars:
        text = text[:max_chars] + '\n[TRUNCATED]'
 
    if not text.strip():
        print(f"  ✗ No text extracted from pages {page_indices}")
        return {'status': 'no_text'}
 
    # Call GPT-4o
    data, api_note = call_gpt(text, unit=unit)
 
    if api_note == 'api_limit_reached':
        print(f"  ⚠ API limit reached")
        return {'status': 'api_limit'}
 
    if not data:
        print(f"  ✗ GPT-4o failed: {api_note}")
        upsert(ticker, isin_map.get(ticker), period, period_end_date,
               filename, company_type, {}, 0.0, True, f'repair_failed:{api_note}')
        return {'status': 'gpt_failed', 'note': api_note}
 
    # Validate
    data, conf, needs_review, val_notes = validate(data, company_type)
    final_notes = f'repair_run | {val_notes}' if api_note == 'OK' else f'repair_run | {api_note} | {val_notes}'
 
    # Save
    upsert(ticker, isin_map.get(ticker), period, period_end_date,
           filename, company_type, data, conf, needs_review, final_notes)
 
    icon = '✓' if not needs_review else '⚠'
    print(f"  {icon} {ticker:<28} {period}  conf={conf:.2f}  [{_requests_today} req]  {val_notes[:50]}")
    time.sleep(DELAY)
 
    return {'status': 'ok', 'confidence': conf, 'needs_review': needs_review}
 
 
print("✓ Cell 4 OK — repair_pdf() defined")

✓ Cell 4 OK — repair_pdf() defined


In [10]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — DIAGNOSE before repairing                                     ║
# ║                                                                          ║
# ║  Run this cell first to confirm which pages to send for each company.   ║
# ║  Read the output, then update page_indices in Cell 6.                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
# Diagnose one representative PDF per failing company
# This uses NO API requests — pure local pdfplumber analysis
 
print("Running diagnostics on failing PDFs...")
print("(No API calls — this is free)\n")
 
DIAGNOSE_LIST = [
    ('STB',            'stb_efd311216.pdf'),
    ('UIB',            'uib_efd311216.pdf'),
    ('ONE TECH HOLDING','oth_efd311218.pdf'),
    ('TELNET HOLDING', 'telnet_efd311216.pdf'),
    ('SOTUMAG',        'sotumag_efd311217.pdf'),
    ('SOPAT',          'sopat_efd_2022.pdf'),
    ('ADWYA',          'adwya_efd311222.pdf'),
    ('CELLCOM',        'cellcom_efd311219.pdf'),
    ('ICF',            'icf_efd_2018.pdf'),
    ('SPDIT - SICAF',  'spdit_sicaf_efd311216.pdf'),
    ('ASSU MAGHREBIA VIE', 'maghrebia_vie_efd311218.pdf'),
]
 
for ticker, fname in DIAGNOSE_LIST:
    diagnose_pdf(ticker, fname)
 
print("\n✓ Cell 5 complete — read the output above to determine page_indices")
print("  Look for pages labeled ACTIF + TOTAL+NUMS — those are your balance sheet pages")
print("  Look for pages labeled RESULTAT + TOTAL+NUMS — those are your P&L pages")

Running diagnostics on failing PDFs...
(No API calls — this is free)


DIAGNOSIS: STB / stb_efd311216.pdf
Total pages : 55

Unit declarations (first 6 pages):
  Page 1: (En 1.000 DT)
  Page 2: (unité : en 1000DT)
  Page 3: (En 1.000 DT)
  Page 4: (Unité : en milliers de dinars)
  Page 5: 155 375 000 actions d'une valeur de 5 dinars chacune, admise à la côte permanent

Pages with financial content:
  Page  1: BILAN, ACTIF, PASSIF, BCT_CODES
  Page  2: BILAN, ACTIF, PASSIF
  Page  3: BILAN, PASSIF, PNB
  Page  5: BILAN
  Page  6: BILAN, ACTIF
  Page  7: BILAN, ACTIF, PASSIF
  Page  8: ACTIF, CA
  Page  9: BILAN
  Page 10: BILAN
  Page 11: BILAN
  Page 13: PASSIF
  Page 29: ACTIF
  Page 30: ACTIF
  Page 31: ACTIF
  Page 38: BILAN, PASSIF
  Page 39: PASSIF
  Page 40: TOTAL+NUMS
  Page 41: BILAN, PASSIF
  Page 42: BILAN
  Page 44: BILAN, PASSIF
  Page 47: BILAN
  Page 48: BILAN, ACTIF, PASSIF, BCT_CODES
  Page 49: BILAN, ACTIF, PASSIF
  Page 52: TOTAL+NUMS
  Page 53: ACTIF
  Page 54: ACTIF,

In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — REPAIR: Category A — Wrong unit (DT companies)               ║
# ║                                                                          ║
# ║  Companies: SOTUMAG, SOPAT, ADWYA, CELLCOM                              ║
# ║  Problem: detect_unit() matched kDT or defaulted to kDT, but the        ║
# ║  PDF uses full DT ('chiffres arrondis au dinar tunisien')                ║
# ║  Fix: force unit='DT', use pages confirmed by Cell 5 diagnostics        ║
# ║                                                                          ║
# ║  UPDATE page_indices BASED ON Cell 5 output before running!             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
isin_map = get_isin_map()
print(f"ISIN map: {len(isin_map)} stocks loaded")
print()
 
# ── SOTUMAG ────────────────────────────────────────────────────────────────
# From PDF inspection: BILAN on page 1 (index 0) and 2 (index 1)
# Unit: "chiffres arrondis au dinar tunisien" = full DT
# UPDATE page_indices below if Cell 5 shows different pages!
 
SOTUMAG_REPAIRS = [
    # (filename, period, period_end_date)
    # NULL DATE file handled separately in pgAdmin
    ('sotumag_efd311217.pdf',          'FY 2017', '2017-12-31'),
    ('sotumag_efd311218.pdf',          'FY 2018', '2018-12-31'),
    ('sotumag_efd311219.pdf',          'FY 2019', '2019-12-31'),
    ('sotumag_efd_2020_23-06-21.pdf',  'FY 2020', '2020-12-31'),
    ('sotumag_efd_2021_22-06-22.pdf',  'FY 2021', '2021-12-31'),
    ('sotumag_efd311222.pdf',          'FY 2022', '2022-12-31'),
    ('sotumag_efd311223.pdf',          'FY 2023', '2023-12-31'),
    ('sotumag_efd311224.pdf',          'FY 2024', '2024-12-31'),
]
 
print("="*60)
print("REPAIRING SOTUMAG (unit=DT, pages 0+1)")
print("="*60)
for fname, period, ped in SOTUMAG_REPAIRS:
    repair_pdf(
        ticker='SOTUMAG', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',          # CONFIRMED: full dinars
        company_type='non_bank',
        page_indices=[0, 1, 2, 3],  # pages 1-4: cover+BILAN+liabilities+P&L
        isin_map=isin_map
    )
 
# ── SOPAT ──────────────────────────────────────────────────────────────────
# Small food company — run Cell 5 diagnosis first to confirm pages
# Update page_indices based on Cell 5 output
 
SOPAT_REPAIRS = [
    ('sopat_efd_2022.pdf',  'FY 2022', '2022-12-31'),
    ('sopat_efd311223.pdf', 'FY 2023', '2023-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING SOPAT (unit=DT — confirm pages with Cell 5)")
print("="*60)
for fname, period, ped in SOPAT_REPAIRS:
    repair_pdf(
        ticker='SOPAT', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        page_indices=[0, 1, 2, 3, 4],  # try first 5 — update after Cell 5
        isin_map=isin_map
    )
 
# ── ADWYA ──────────────────────────────────────────────────────────────────
ADWYA_REPAIRS = [
    ('adwya_efd311222.pdf', 'FY 2022', '2022-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING ADWYA FY2022")
print("="*60)
for fname, period, ped in ADWYA_REPAIRS:
    repair_pdf(
        ticker='ADWYA', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        page_indices=[0, 1, 2, 3, 4],
        isin_map=isin_map
    )
 
# ── CELLCOM ────────────────────────────────────────────────────────────────
CELLCOM_REPAIRS = [
    ('cellcom_efd311219.pdf', 'FY 2019', '2019-12-31'),
    ('cellcom_efd311220.pdf', 'FY 2020', '2020-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING CELLCOM FY2019/2020")
print("="*60)
for fname, period, ped in CELLCOM_REPAIRS:
    repair_pdf(
        ticker='CELLCOM', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        page_indices=[0, 1, 2, 3, 4],
        isin_map=isin_map
    )
 
print("\n✓ Cell 6 complete")

ISIN map: 70 stocks loaded

REPAIRING SOTUMAG (unit=DT, pages 0+1)
  ✓ SOTUMAG                      FY 2017  conf=0.62  [2 req]  OK
  ✓ SOTUMAG                      FY 2018  conf=0.62  [3 req]  OK
  ✓ SOTUMAG                      FY 2019  conf=0.62  [4 req]  OK
  ✓ SOTUMAG                      FY 2020  conf=0.62  [5 req]  OK
  ✓ SOTUMAG                      FY 2021  conf=0.62  [6 req]  OK
  ✓ SOTUMAG                      FY 2022  conf=0.62  [7 req]  OK
  ✓ SOTUMAG                      FY 2023  conf=0.62  [8 req]  OK
  ✓ SOTUMAG                      FY 2024  conf=0.62  [9 req]  OK

REPAIRING SOPAT (unit=DT — confirm pages with Cell 5)
  ⚠ SOPAT                        FY 2022  conf=0.00  [10 req]  total_assets:missing
  ⚠ SOPAT                        FY 2023  conf=0.00  [11 req]  total_assets:missing

REPAIRING ADWYA FY2022
  ⚠ ADWYA                        FY 2022  conf=0.00  [12 req]  total_assets:missing

REPAIRING CELLCOM FY2019/2020
  ⚠ CELLCOM                      FY 2019  conf=0.00

In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — REPAIR: Category B — Non-standard page layout                ║
# ║                                                                          ║
# ║  Companies: STB, UIB, ONE TECH HOLDING, TELNET HOLDING                  ║
# ║  Problem: BILAN is on a different page than our scanner expects          ║
# ║  STB/UIB: banks with BILAN on page 1 (index 0) — BCT standard format   ║
# ║  ONE TECH/TELNET: consolidated reports, BILAN on page 5-8               ║
# ║                                                                          ║
# ║  IMPORTANT: Run Cell 5 first and update page_indices for each company!  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
# ── STB ────────────────────────────────────────────────────────────────────
# STB uses the BCT standard format with BILAN on page 1 (index 0)
# The BILAN keyword + numbers are right at the start of the document
# UPDATE page_indices based on Cell 5 diagnosis output
 
STB_REPAIRS = [
    ('stb_efd311216.pdf',   'FY 2016', '2016-12-31'),
    ('stb_efd311218.pdf',   'FY 2018', '2018-12-31'),
    ('efd_2019_stb.pdf',    'FY 2019', '2019-12-31'),
    ('efd_2020_stb.pdf',    'FY 2020', '2020-12-31'),
    ('stb_efd311221.pdf',   'FY 2021', '2021-12-31'),
    ('stb_efd311222.pdf',   'FY 2022', '2022-12-31'),
    ('stb_efd311223.pdf',   'FY 2023', '2023-12-31'),
    # stb_efd311217_0.pdf → NULL DATE → fixed in pgAdmin
    # stb_efd311224.pdf → already skipped (conf >= 0.5)
]
 
print("="*60)
print("REPAIRING STB (bank, kDT, pages 0+1+2)")
print("="*60)
for fname, period, ped in STB_REPAIRS:
    repair_pdf(
        ticker='STB', filename=fname,
        period=period, period_end_date=ped,
        unit='kDT',      # STB is a bank → kDT
        company_type='bank',
        page_indices=[0, 1, 2, 3],  # BILAN on page 1 (index 0), P&L on page 2-3
        isin_map=isin_map
    )
 
# ── UIB ────────────────────────────────────────────────────────────────────
UIB_REPAIRS = [
    ('uib_efd311216.pdf',  'FY 2016', '2016-12-31'),
    ('uib_efd311217.pdf',  'FY 2017', '2017-12-31'),
    ('uib_efd311218.pdf',  'FY 2018', '2018-12-31'),
    ('efd_2019_uib.pdf',   'FY 2019', '2019-12-31'),
    ('uib_efd311220.pdf',  'FY 2020', '2020-12-31'),
    ('uib_efd311221.pdf',  'FY 2021', '2021-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING UIB (bank, kDT, pages 0+1+2)")
print("="*60)
for fname, period, ped in UIB_REPAIRS:
    repair_pdf(
        ticker='UIB', filename=fname,
        period=period, period_end_date=ped,
        unit='kDT',
        company_type='bank',
        page_indices=[0, 1, 2, 3],
        isin_map=isin_map
    )
 
# ── ONE TECH HOLDING ───────────────────────────────────────────────────────
# Consolidated holding company — BILAN typically on page 5-8
# UPDATE page_indices after running Cell 5 diagnosis
 
OTH_REPAIRS = [
    ('oth_efd311218.pdf', 'FY 2018', '2018-12-31'),
    ('oth_efd311220.pdf', 'FY 2020', '2020-12-31'),
    ('oth_efd311221.pdf', 'FY 2021', '2021-12-31'),
    ('oth_efd311222.pdf', 'FY 2022', '2022-12-31'),
    ('oth_efd311224.pdf', 'FY 2024', '2024-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING ONE TECH HOLDING (non_bank, DT, wide page scan)")
print("="*60)
for fname, period, ped in OTH_REPAIRS:
    repair_pdf(
        ticker='ONE TECH HOLDING', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        # Wider scan: pages 0-9 to catch BILAN wherever it is
        page_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        isin_map=isin_map,
        max_chars=10000  # allow more text since we scan more pages
    )
 
# ── TELNET HOLDING ─────────────────────────────────────────────────────────
TELNET_REPAIRS = [
    ('telnet_efd311216.pdf',         'FY 2016', '2016-12-31'),
    ('telnet_holding_efd311217.pdf', 'FY 2017', '2017-12-31'),
    ('telnet_holding_efd311218.pdf', 'FY 2018', '2018-12-31'),
    ('telnet_holding_efd311219.pdf', 'FY 2019', '2019-12-31'),
    ('telnet_holding_efd311221.pdf', 'FY 2021', '2021-12-31'),
    ('telnet_holding_efd311222.pdf', 'FY 2022', '2022-12-31'),
    ('telnet_holding_efd311224.pdf', 'FY 2024', '2024-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING TELNET HOLDING (non_bank, DT, wide page scan)")
print("="*60)
for fname, period, ped in TELNET_REPAIRS:
    repair_pdf(
        ticker='TELNET HOLDING', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        page_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        isin_map=isin_map,
        max_chars=10000
    )
 
print("\n✓ Cell 7 complete")

REPAIRING STB (bank, kDT, pages 0+1+2)
  ✓ STB                          FY 2016  conf=0.88  [15 req]  OK
  ✓ STB                          FY 2018  conf=0.88  [16 req]  OK
  ✓ STB                          FY 2019  conf=0.88  [17 req]  OK
  ✓ STB                          FY 2020  conf=0.88  [18 req]  OK
  ✓ STB                          FY 2021  conf=0.88  [19 req]  OK
  ✓ STB                          FY 2022  conf=0.88  [20 req]  OK
  ✓ STB                          FY 2023  conf=0.88  [21 req]  OK

REPAIRING UIB (bank, kDT, pages 0+1+2)
  ✓ UIB                          FY 2016  conf=0.88  [22 req]  OK
  ✓ UIB                          FY 2017  conf=0.88  [23 req]  OK
  ⚠ UIB                          FY 2018  conf=0.00  [24 req]  total_assets:missing
  ✓ UIB                          FY 2019  conf=0.88  [25 req]  OK
  ✓ UIB                          FY 2020  conf=0.88  [26 req]  OK
  ✓ UIB                          FY 2021  conf=0.88  [27 req]  OK

REPAIRING ONE TECH HOLDING (non_bank, DT, wi

In [8]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — REPAIR: Category D — Isolated failures + SPDIT               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
# ── ICF FY 2018 ────────────────────────────────────────────────────────────
print("="*60)
print("REPAIRING ICF FY2018")
print("="*60)
repair_pdf(
    ticker='ICF', filename='icf_efd_2018.pdf',
    period='FY 2018', period_end_date='2018-12-31',
    unit='DT',          # ICF is in LEASING_TICKERS → DT
    company_type='non_bank',
    page_indices=[0, 1, 2, 3, 4, 5],
    isin_map=isin_map
)
 
# ── ASSU MAGHREBIA VIE FY 2018 ─────────────────────────────────────────────
print("\n" + "="*60)
print("REPAIRING ASSU MAGHREBIA VIE FY2018")
print("="*60)
repair_pdf(
    ticker='ASSU MAGHREBIA VIE', filename='maghrebia_vie_efd311218.pdf',
    period='FY 2018', period_end_date='2018-12-31',
    unit='DT',
    company_type='insurance',
    page_indices=[0, 1, 2, 3, 4, 5],
    isin_map=isin_map
)
 
# ── SPDIT - SICAF ──────────────────────────────────────────────────────────
# Investment fund — different balance sheet structure
# conf=0.25 means some fields extracted but total_assets missing
SPDIT_REPAIRS = [
    ('spdit_sicaf_efd311216.pdf',          'FY 2016', '2016-12-31'),
    ('spdit_sicaf_efd311217.pdf',          'FY 2017', '2017-12-31'),
    ('spdit_sicaf_efd311221.pdf',          'FY 2021', '2021-12-31'),
    ('spdit_sicaf_efd311222.pdf',          'FY 2022', '2022-12-31'),
]
 
print("\n" + "="*60)
print("REPAIRING SPDIT - SICAF (investment fund, wide scan)")
print("="*60)
for fname, period, ped in SPDIT_REPAIRS:
    repair_pdf(
        ticker='SPDIT - SICAF', filename=fname,
        period=period, period_end_date=ped,
        unit='DT',
        company_type='non_bank',
        page_indices=[0, 1, 2, 3, 4, 5, 6],
        isin_map=isin_map
    )
 
# ── BNA ASSURANCES ─────────────────────────────────────────────────────────
# conf=0.25 — equity was extracted but total_assets missing
print("\n" + "="*60)
print("REPAIRING BNA ASSURANCES FY2024")
print("="*60)
repair_pdf(
    ticker='BNA ASSURANCES', filename='bna_assurances_efd311224_0.pdf',
    period='FY 2024', period_end_date='2024-12-31',
    unit='DT',
    company_type='insurance',
    page_indices=[0, 1, 2, 3, 4, 5, 6],
    isin_map=isin_map
)
 
print("\n✓ Cell 8 complete")

REPAIRING ICF FY2018
  ⚠ ICF                          FY 2018  conf=0.00  [40 req]  total_assets:missing

REPAIRING ASSU MAGHREBIA VIE FY2018
  ⚠ ASSU MAGHREBIA VIE           FY 2018  conf=0.00  [41 req]  total_assets:missing

REPAIRING SPDIT - SICAF (investment fund, wide scan)
  ✓ SPDIT - SICAF                FY 2016  conf=0.62  [42 req]  OK
  ✓ SPDIT - SICAF                FY 2017  conf=0.62  [43 req]  OK
  ✓ SPDIT - SICAF                FY 2021  conf=0.62  [44 req]  OK
  ✓ SPDIT - SICAF                FY 2022  conf=0.62  [45 req]  OK

REPAIRING BNA ASSURANCES FY2024
  ⚠ BNA ASSURANCES               FY 2024  conf=0.00  [46 req]  total_assets:missing

✓ Cell 8 complete


In [9]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Verify: check what was fixed                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
# Companies that were repaired
repaired_tickers = [
    'SOTUMAG', 'SOPAT', 'ADWYA', 'CELLCOM',
    'STB', 'UIB', 'ONE TECH HOLDING', 'TELNET HOLDING',
    'ICF', 'ASSU MAGHREBIA VIE', 'SPDIT - SICAF', 'BNA ASSURANCES'
]
 
placeholders = ','.join(['%s'] * len(repaired_tickers))
df = pd.read_sql(f'''
    SELECT
        ticker,
        COUNT(*) AS total_records,
        COUNT(*) FILTER (WHERE total_assets IS NOT NULL) AS has_assets,
        ROUND(AVG(extraction_confidence)::numeric, 3) AS avg_conf,
        COUNT(*) FILTER (WHERE extraction_confidence >= 0.5) AS good_records,
        COUNT(*) FILTER (WHERE needs_review = TRUE) AS needs_review
    FROM financial_statements
    WHERE ticker IN ({placeholders})
    GROUP BY ticker
    ORDER BY ticker
''', conn, params=repaired_tickers)
 
conn.close()
 
print("Repair results by company:")
print(df.to_string(index=False))
print()
print(f"Total requests used this session: {_requests_today}")
 
# Summary
still_bad = df[df['has_assets'] == 0]['ticker'].tolist()
if still_bad:
    print(f"\nStill missing total_assets: {still_bad}")
    print("→ Run diagnose_pdf() on these and adjust page_indices")
else:
    print("\n✓ All repaired companies now have total_assets extracted!")

Repair results by company:
            ticker  total_records  has_assets  avg_conf  good_records  needs_review
             ADWYA             12          11     0.563            11             1
ASSU MAGHREBIA VIE             10           9     0.475             9             1
    BNA ASSURANCES              2           1     0.188             0             1
           CELLCOM             16          14     0.570            14            13
               ICF              6           5     0.583             5             2
  ONE TECH HOLDING              7           2     0.143             2             6
             SOPAT              8           6     0.469             6             8
           SOTUMAG              8           8     0.625             8             0
     SPDIT - SICAF              8           8     0.563             8             0
               STB              9           9     0.833             9             0
    TELNET HOLDING              7           7    

C:\Users\Negza\AppData\Local\Temp\ipykernel_23160\4189879714.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f'''


In [11]:
# ── Re-extract missing ONE TECH HOLDING years ──────────────────────────
print("ONE TECH HOLDING — missing FY2018/2020/2021/2022/2024")
for fname, period, ped in [
    ('oth_efd311218.pdf', 'FY 2018', '2018-12-31'),
    ('oth_efd311220.pdf', 'FY 2020', '2020-12-31'),
    ('oth_efd311221.pdf', 'FY 2021', '2021-12-31'),
    ('oth_efd311222.pdf', 'FY 2022', '2022-12-31'),
    ('oth_efd311224.pdf', 'FY 2024', '2024-12-31'),
]:
    repair_pdf('ONE TECH HOLDING', fname, period, ped,
               unit='DT', company_type='non_bank',
               page_indices=[6, 7, 8, 9, 10],
               isin_map=isin_map)

# ── Re-extract missing UIB years ───────────────────────────────────────
print("\nUIB — missing FY2018/2021")
for fname, period, ped in [
    ('uib_efd311218.pdf', 'FY 2018', '2018-12-31'),
    ('uib_efd311221.pdf', 'FY 2021', '2021-12-31'),
]:
    repair_pdf('UIB', fname, period, ped,
               unit='kDT', company_type='bank',
               page_indices=[0, 1, 2],
               isin_map=isin_map)

# ── Re-extract BNA ASSURANCES FY2024 ──────────────────────────────────
print("\nBNA ASSURANCES — FY2024")
repair_pdf('BNA ASSURANCES', 'bna_assurances_efd311224_0.pdf',
           'FY 2024', '2024-12-31',
           unit='DT', company_type='insurance',
           page_indices=[0, 1, 2, 3, 4, 5, 6],
           isin_map=isin_map)

print(f"\nDone — {_requests_today} total API requests used")

ONE TECH HOLDING — missing FY2018/2020/2021/2022/2024
  ⚠ ONE TECH HOLDING             FY 2018  conf=0.00  [47 req]  total_assets:missing
  ⚠ ONE TECH HOLDING             FY 2020  conf=0.00  [48 req]  total_assets:missing
  ⚠ ONE TECH HOLDING             FY 2021  conf=0.00  [49 req]  total_assets:missing
  ⚠ ONE TECH HOLDING             FY 2022  conf=0.00  [50 req]  total_assets:missing

  [Rate limit — waiting 60s attempt 1/3]

KeyboardInterrupt: 